In [0]:
USE CATALOG online_retail;

SELECT COUNT(*) AS bronze_row_count
FROM bronze.online_retail_raw;

SELECT *
FROM bronze.online_retail_raw
LIMIT 10;

bronze_row_count
541909


InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,_rescued_data
536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,01-12-2010 08:26,2.55,17850,United Kingdom,null
536365,71053,WHITE METAL LANTERN,6,01-12-2010 08:26,3.39,17850,United Kingdom,null
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,01-12-2010 08:26,2.75,17850,United Kingdom,null
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,01-12-2010 08:26,3.39,17850,United Kingdom,null
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,01-12-2010 08:26,3.39,17850,United Kingdom,null
536365,22752,SET 7 BABUSHKA NESTING BOXES,2,01-12-2010 08:26,7.65,17850,United Kingdom,null
536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,01-12-2010 08:26,4.25,17850,United Kingdom,null
536366,22633,HAND WARMER UNION JACK,6,01-12-2010 08:28,1.85,17850,United Kingdom,null
536366,22632,HAND WARMER RED POLKA DOT,6,01-12-2010 08:28,1.85,17850,United Kingdom,null
536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,01-12-2010 08:34,1.69,13047,United Kingdom,null


In [0]:
CREATE OR REPLACE TABLE online_retail.gold.product_sales AS

SELECT
    StockCode,
    Description,
    SUM(Quantity) AS total_quantity_sold,
    COUNT(DISTINCT InvoiceNo) AS total_orders,
    ROUND(SUM(Quantity * UnitPrice), 2) AS total_revenue,
    ROUND(
        SUM(Quantity * UnitPrice) / COUNT(DISTINCT InvoiceNo),
        2
    ) AS average_order_value

FROM online_retail.silver.online_retail_clean

GROUP BY
    StockCode,
    Description;
SELECT COUNT(*) AS product_count
FROM online_retail.gold.product_sales;

num_affected_rows,num_inserted_rows


product_count
4158


In [0]:
SELECT
    COUNT(*) AS total_rows,
    COUNT_IF(total_revenue < 0) AS negative_revenue,
    COUNT_IF(total_quantity_sold <= 0) AS invalid_quantity,
    COUNT_IF(total_orders <= 0) AS invalid_orders
FROM online_retail.gold.product_sales;

total_rows,negative_revenue,invalid_quantity,invalid_orders
4158,0,0,0


In [0]:
SELECT
    'sales_summary' AS table_name,
    COUNT(*) AS row_count
FROM online_retail.gold.sales_summary

UNION ALL

SELECT
    'country_sales',
    COUNT(*)
FROM online_retail.gold.country_sales

UNION ALL

SELECT
    'product_sales',
    COUNT(*)
FROM online_retail.gold.product_sales;

table_name,row_count
sales_summary,305
country_sales,38
product_sales,4158


In [0]:
CREATE OR REPLACE TABLE online_retail.gold.product_sales AS
SELECT
    StockCode,
    Description,
    SUM(Quantity) AS total_quantity_sold,
    COUNT(DISTINCT InvoiceNo) AS total_orders,
    ROUND(SUM(Quantity * UnitPrice), 2) AS total_revenue,
    ROUND(
        SUM(Quantity * UnitPrice) / COUNT(DISTINCT InvoiceNo),
        2
    ) AS average_order_value
FROM online_retail.silver.online_retail_clean
GROUP BY
    StockCode,
    Description;

num_affected_rows,num_inserted_rows


In [0]:
SELECT *
FROM online_retail.gold.product_sales
ORDER BY total_revenue DESC
LIMIT 20;

StockCode,Description,total_quantity_sold,total_orders,total_revenue,average_order_value
DOT,DOTCOM POSTAGE,706,706,206248.77,292.14
22423,REGENCY CAKESTAND 3 TIER,13851,1988,174156.54,87.60
23843,"PAPER CRAFT , LITTLE BIRDIE",80995,1,168469.60,168469.60
85123A,WHITE HANGING HEART T-LIGHT HOLDER,37580,2189,104284.24,47.64
47566,PARTY BUNTING,18283,1685,99445.23,59.02
85099B,JUMBO BAG RED RETROSPOT,48371,2089,94159.81,45.07
23166,MEDIUM CERAMIC TOP STORAGE JAR,78033,247,81700.92,330.77
POST,POSTAGE,3150,1126,78101.88,69.36
M,Manual,6984,289,77750.27,269.03
23084,RABBIT NIGHT LIGHT,30739,994,66870.03,67.27


In [0]:
SELECT COUNT(*) AS product_count
FROM online_retail.gold.product_sales;

product_count
4158


In [0]:
-- ============================================
-- FINAL PIPELINE VALIDATION
-- ============================================

SELECT
    'Bronze' AS layer,
    'online_retail_raw' AS table_name,
    COUNT(*) AS row_count
FROM online_retail.bronze.online_retail_raw

UNION ALL

SELECT
    'Silver',
    'online_retail_clean',
    COUNT(*)
FROM online_retail.silver.online_retail_clean

UNION ALL

SELECT
    'Gold',
    'sales_summary',
    COUNT(*)
FROM online_retail.gold.sales_summary

UNION ALL

SELECT
    'Gold',
    'country_sales',
    COUNT(*)
FROM online_retail.gold.country_sales

UNION ALL

SELECT
    'Gold',
    'product_sales',
    COUNT(*)
FROM online_retail.gold.product_sales;

layer,table_name,row_count
Bronze,online_retail_raw,541909
Silver,online_retail_clean,524878
Gold,sales_summary,305
Gold,country_sales,38
Gold,product_sales,4158


In [0]:
-- ============================================
-- FINAL DATA QUALITY CHECK
-- ============================================

SELECT
    COUNT(*) AS total_rows,
    COUNT_IF(total_revenue < 0) AS negative_revenue,
    COUNT_IF(total_orders <= 0) AS invalid_orders,
    COUNT_IF(total_quantity_sold <= 0) AS invalid_quantity
FROM online_retail.gold.product_sales;

total_rows,negative_revenue,invalid_orders,invalid_quantity
4158,0,0,0


In [0]:
-- ============================================
-- OVERALL BUSINESS SUMMARY
-- ============================================

SELECT
    MIN(sales_date) AS first_sales_date,
    MAX(sales_date) AS last_sales_date,
    SUM(total_orders) AS total_orders,
    SUM(total_revenue) AS total_revenue
FROM online_retail.gold.sales_summary;

first_sales_date,last_sales_date,total_orders,total_revenue
2010-12-01,2011-12-09,19960,10642110.80
